In [ ]:
import os
import shutil
import json

# =========================================================
# 1. SETUP & PATHS
# =========================================================
# Kaggle Working directory cleanup
print("🧹 Cleaning workspace...")
%cd /kaggle/working
!rm -rf *
!git clone https://github.com/lpcvai/26LPCVC_Track2_Sample_Solution.git
%cd 26LPCVC_Track2_Sample_Solution
!pip install -r requirements.txt
!pip install qai-hub

# Path Definitions
STAGING_DIR = '/tmp/qevd_staging'
RAW_EXTRACT_DIR = '/tmp/qevd_raw'
FINAL_DATASET_DIR = '/kaggle/working/qevd_final'
REPO_DIR = '/kaggle/working/26LPCVC_Track2_Sample_Solution'

os.makedirs(STAGING_DIR, exist_ok=True)
os.makedirs(FINAL_DATASET_DIR, exist_ok=True)

# =========================================================
# 2. GET LABELS & REPO
# =========================================================
print("📥 Cloning repo for labels...")
!git clone https://github.com/lpcvai/26LPCVC_Track2_Sample_Solution.git
LABEL_JSON = os.path.join(REPO_DIR, 'fine_grained_labels_release.json')

# =========================================================
# 3. DOWNLOAD & EXTRACT (Part 1)
# =========================================================
print("📥 Downloading QEVD Part 1...")
BASE_URL = "https://softwarecenter.qualcomm.com/api/download/software/dataset/AIDataset/Qualcomm_Exercise_Video_Dataset/QEVD-FIT-300k-Part-1"

# Downloading to /tmp to save /working space
!wget -q -O {STAGING_DIR}/p1.zip "{BASE_URL}/QEVD-FIT-300k-Part-1.zip"
!wget -q -O {STAGING_DIR}/p1.z01 "{BASE_URL}/QEVD-FIT-300k-Part-1.z01"
!wget -q -O {STAGING_DIR}/p1.z02 "{BASE_URL}/QEVD-FIT-300k-Part-1.z02"

print("📦 Extracting archives to /tmp...")
# 7z extracts split volumes automatically
!7z x {STAGING_DIR}/p1.zip -o{RAW_EXTRACT_DIR} -y > /dev/null

# =========================================================
# 4. ORGANIZE VIDEOS BY CLASS
# =========================================================
print("📂 Organizing videos into classes...")
with open(LABEL_JSON, 'r') as f:
    labels_data = json.load(f)

# Extraction creates a subfolder usually named after the zip
actual_raw_path = os.path.join(RAW_EXTRACT_DIR, 'QEVD-FIT-300k-Part-1')
count = 0

if os.path.exists(actual_raw_path):
    for entry in labels_data:
        v_path = entry.get('video_path', '')
        filename = os.path.basename(v_path)
        src = os.path.join(actual_raw_path, filename)
        
        if os.path.exists(src):
            # Clean label name for folder
            exercise_label = entry.get('label', 'unknown').strip().replace(" ", "_").replace("/", "-")
            dest_dir = os.path.join(FINAL_DATASET_DIR, exercise_label)
            
            if not os.path.exists(dest_dir):
                os.makedirs(dest_dir)
            
            # MOVE from /tmp to /working/qevd_final
            shutil.move(src, os.path.join(dest_dir, filename))
            count += 1
            if count % 2000 == 0:
                print(f"✅ Moved {count} videos...")

print(f"⭐ Organization complete! Total videos: {count}")

# =========================================================
# 5. FINAL CLEANUP (Crucial for Dataset Creation)
# =========================================================
print("🧹 Final cleanup of Repo and Temporary files...")

# Remove the cloned repo to keep output clean
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Remove extraction staging areas
!rm -rf {STAGING_DIR}
!rm -rf {RAW_EXTRACT_DIR}

print("\n🚀 SUCCESS!")
print(f"Your dataset is ready at: {FINAL_DATASET_DIR}")
print("You can now click 'Save Version' -> 'Quick Save' and then create a Kaggle Dataset from the output folder.")


🧹 Cleaning workspace...
/kaggle/working
Cloning into '26LPCVC_Track2_Sample_Solution'...
remote: Enumerating objects: 525, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 525 (delta 33), reused 28 (delta 28), pack-reused 485 (from 2)
Receiving objects: 100% (525/525), 1.95 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/kaggle/working/26LPCVC_Track2_Sample_Solution
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 71.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 40.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

: 